# Aula 18 - Grafos de Logística de Insumos e Produtos Acabados — Linha de Paçoca

**Disciplina:** ECAA08 — Automática (2026.2) — UNIFEI  
**Projeto:** SCADA-Core Automática / Linha de Produção de Paçoca  
**Equipe:** Grupo 7  
**Perfil:** Engenharia de Controle e Automação (Matemática Discreta & Teoria dos Grafos)  

---

Este notebook complementa a malha de processo das aulas anteriores (Aulas 11 a 17) com duas redes do layout logístico da **Fábrica de Paçoca do Grupo 7**: um dígrafo de fluxo de materiais ($G_M$, do recebimento de insumos até a expedição do produto acabado) e outro para a circulação interna de veículos ($G_V$). Os pesos são estimativas didáticas em metros, não medições de campo.

In [1]:
from dataclasses import dataclass
from heapq import heappop, heappush
from math import inf
from typing import Dict, List, Tuple


@dataclass(frozen=True)
class Rota:
    destino: str
    peso: float
    descricao: str


class GrafoLogistico:
    def __init__(self) -> None:
        self._adjacencias: Dict[str, List[Rota]] = {}

    def adicionar_rota(self, origem: str, destino: str, peso: float, descricao: str) -> None:
        if peso < 0:
            raise ValueError("Dijkstra exige pesos não negativos.")
        self._adjacencias.setdefault(origem, []).append(Rota(destino, peso, descricao))
        self._adjacencias.setdefault(destino, [])

    def menor_caminho(self, origem: str, destino: str) -> Tuple[float, List[str]]:
        if origem not in self._adjacencias or destino not in self._adjacencias:
            raise KeyError("Origem e destino devem pertencer ao grafo.")

        distancias = {vertice: inf for vertice in self._adjacencias}
        predecessores: Dict[str, str] = {}
        distancias[origem] = 0.0
        fila: List[Tuple[float, str]] = [(0.0, origem)]

        while fila:
            distancia_atual, atual = heappop(fila)
            if distancia_atual != distancias[atual]:
                continue
            if atual == destino:
                break
            for rota in self._adjacencias[atual]:
                candidata = distancia_atual + rota.peso
                if candidata < distancias[rota.destino]:
                    distancias[rota.destino] = candidata
                    predecessores[rota.destino] = atual
                    heappush(fila, (candidata, rota.destino))

        if distancias[destino] == inf:
            raise ValueError(f"Não existe rota de {origem} para {destino}.")

        caminho = [destino]
        while caminho[-1] != origem:
            caminho.append(predecessores[caminho[-1]])
        caminho.reverse()
        return distancias[destino], caminho


def adicionar_rotas(grafo: GrafoLogistico, rotas: List[Tuple[str, str, float, str]]) -> None:
    for origem, destino, peso, descricao in rotas:
        grafo.adicionar_rota(origem, destino, peso, descricao)


# GM: fluxo de materiais da Fábrica de Paçoca (Grupo 7).
# Os pesos são distâncias internas estimadas para fins didáticos.
fluxo_materiais = GrafoLogistico()
adicionar_rotas(fluxo_materiais, [
    ("Portaria", "Balança de entrada", 25, "liberação e pesagem do recebimento"),
    ("Balança de entrada", "Recebimento / Galpão A", 55, "direcionamento à descarga"),
    ("Recebimento / Galpão A", "Box: amendoim cru", 18, "armazenagem do amendoim cru recebido"),
    ("Recebimento / Galpão A", "Box: açúcar", 20, "armazenagem do açúcar recebido"),
    ("Recebimento / Galpão A", "Box: sal", 22, "armazenagem do sal"),
    ("Recebimento / Galpão A", "Box: embalagens e aditivos", 25, "armazenagem de embalagens e aditivos"),
    ("Recebimento / Galpão A", "Tanque de glucose", 28, "recebimento de glucose/xarope líquido"),
    ("Box: amendoim cru", "Moega de recepção", 35, "transferência por esteira"),
    ("Box: açúcar", "Moega de recepção", 32, "transferência por esteira"),
    ("Box: sal", "Moega de recepção", 30, "transferência por esteira"),
    ("Box: embalagens e aditivos", "Moega de recepção", 38, "transferência por esteira"),
    ("Moega de recepção", "Silos de dosagem", 40, "alimentação dos dosadores"),
    ("Tanque de glucose", "Silos de dosagem", 45, "dosagem do xarope ligante"),
    ("Silos de dosagem", "Moinho / Torrador", 18, "torra e moagem do amendoim dosado"),
    ("Moinho / Torrador", "Homogeneização (HOM-301)", 20, "formação da massa de paçoca"),
    ("Homogeneização (HOM-301)", "Prensagem (PRN-401)", 32, "compactação da massa em barras/tabletes"),
    ("Prensagem (PRN-401)", "Ensacamento / paletização", 48, "corte, resfriamento e embalagem"),
    ("Ensacamento / paletização", "Estoque Paçoca Tradicional", 35, "endereçamento do produto acabado"),
    ("Ensacamento / paletização", "Estoque Paçoca Zero Açúcar", 42, "endereçamento do produto acabado"),
    ("Ensacamento / paletização", "Estoque Paçoca Premium", 50, "endereçamento do produto acabado"),
    ("Estoque Paçoca Tradicional", "Docas de expedição", 28, "separação para carregamento"),
    ("Estoque Paçoca Zero Açúcar", "Docas de expedição", 25, "separação para carregamento"),
    ("Estoque Paçoca Premium", "Docas de expedição", 32, "separação para carregamento"),
])

distancia, caminho = fluxo_materiais.menor_caminho("Portaria", "Docas de expedição")
print("Fluxo de materiais até a expedição")
print("  " + " -> ".join(caminho))
print(f"  Distância didática total: {distancia:.0f} m")
assert "Silos de dosagem" in caminho
assert "Ensacamento / paletização" in caminho

print("\nRotas dos insumos até a expedição")
for insumo in [
    "Box: amendoim cru",
    "Box: açúcar",
    "Box: sal",
    "Box: embalagens e aditivos",
    "Tanque de glucose",
]:
    distancia_insumo, caminho_insumo = fluxo_materiais.menor_caminho(insumo, "Docas de expedição")
    print(f"  {insumo}: {distancia_insumo:.0f} m ({' -> '.join(caminho_insumo)})")


# GV: circulação interna. Cada direção permitida é uma aresta explícita.
circulacao_veiculos = GrafoLogistico()
anel_viario = [
    ("Portaria", "Balança de entrada", 25, "entrada de veículos"),
    ("Balança de entrada", "Pátio de recebimento", 55, "acesso à descarga"),
    ("Pátio de recebimento", "Galpão A", 40, "rota de descarga"),
    ("Galpão A", "Área de processamento", 65, "rota interna de serviço"),
    ("Área de processamento", "Galpão B", 75, "rota para armazenagem"),
    ("Galpão B", "Docas de expedição", 30, "rota de carregamento"),
    ("Docas de expedição", "Balança de saída", 60, "saída de veículos"),
    ("Balança de saída", "Portaria", 35, "liberação de saída"),
]
adicionar_rotas(circulacao_veiculos, anel_viario)

distancia_saida, rota_saida = circulacao_veiculos.menor_caminho(
    "Docas de expedição", "Portaria"
)
print("\nCirculação de veículo após o carregamento")
print("  " + " -> ".join(rota_saida))
print(f"  Distância didática total: {distancia_saida:.0f} m")
assert rota_saida == ["Docas de expedição", "Balança de saída", "Portaria"]


Fluxo de materiais até a expedição
  Portaria -> Balança de entrada -> Recebimento / Galpão A -> Tanque de glucose -> Silos de dosagem -> Moinho / Torrador -> Homogeneização (HOM-301) -> Prensagem (PRN-401) -> Ensacamento / paletização -> Estoque Paçoca Tradicional -> Docas de expedição
  Distância didática total: 334 m

Rotas dos insumos até a expedição
  Box: amendoim cru: 256 m (Box: amendoim cru -> Moega de recepção -> Silos de dosagem -> Moinho / Torrador -> Homogeneização (HOM-301) -> Prensagem (PRN-401) -> Ensacamento / paletização -> Estoque Paçoca Tradicional -> Docas de expedição)
  Box: açúcar: 253 m (Box: açúcar -> Moega de recepção -> Silos de dosagem -> Moinho / Torrador -> Homogeneização (HOM-301) -> Prensagem (PRN-401) -> Ensacamento / paletização -> Estoque Paçoca Tradicional -> Docas de expedição)
  Box: sal: 251 m (Box: sal -> Moega de recepção -> Silos de dosagem -> Moinho / Torrador -> Homogeneização (HOM-301) -> Prensagem (PRN-401) -> Ensacamento / paletização -> 